# Clean Data

Convert each source's raw JSON documents (downloaded by `1_download_data.ipynb`) into plain `.txt` files under `processed_sources/<source>/`, preserving the folder structure. Non-JSON files are copied through unchanged; empty or invalid JSON files are skipped and counted. Prints progress every 100 files and skips files already converted, so it's safe to re-run if the kernel stops partway through.

In [4]:
import os
import json
import re
import shutil


sources = ["confluence", "fireflies", "github", "gmail", "google_drive", "hubspot", "jira", "linear", "slack"]

skip_fields = {"_file", "doc_id", "dataset_doc_uuid", "title_field_name", "content_field_names"}


# Strip common markdown syntax (headings, bold/italic, inline code, bullet markers, horizontal
# rules, repeated blank lines) from a string so it reads as plain prose instead of raw markdown.
def clean_markdown(text):

    if not isinstance(text, str):
        return str(text)

    # Remove headings (#, ##, ### ...)
    text = re.sub(r'^\s*#{1,6}\s*', '', text, flags = re.MULTILINE)

    # Remove bold/italic markers
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)

    # Remove inline code
    text = re.sub(r'`([^`]*)`', r'\1', text)

    # Convert markdown bullet symbols
    text = re.sub(r'^\s*[-*]\s+', '- ', text, flags = re.MULTILINE)

    # Replace multiple blank lines
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove horizontal rules
    text = re.sub(r'^\s*([-*_])\1{2,}\s*$', '', text, flags = re.MULTILINE)

    # Remove long runs of dashes or equals
    text = re.sub(r'^\s*[-=]{3,}\s*$', '', text, flags = re.MULTILINE)
    return text.strip()


# Recursively turn a JSON value into indented plain text: strings are markdown-cleaned, lists
# become "- item" bullets, dicts become "key: value" lines, and nesting increases the indent.
def format_value(value, indent = 0):

    prefix = " " * indent

    if value is None:
        return ""

    if isinstance(value, str):
        return clean_markdown(value)

    if isinstance(value, (int, float, bool)):
        return str(value)

    if isinstance(value, list):

        if len(value) == 0:
            return ""

        lines = []

        for item in value:

            if isinstance(item, (dict, list)):
                lines.append(prefix + "-")
                lines.append(format_value(item, indent + 2))
            else:
                lines.append(prefix + "- " + clean_markdown(str(item)))

        return "\n".join(lines)

    if isinstance(value, dict):

        lines = []

        for k, v in value.items():

            if v in ("", None, [], {}):
                continue

            formatted = format_value(v, indent + 2)

            if "\n" in formatted:
                lines.append(prefix + f"{k}:")
                lines.append(formatted)
            else:
                lines.append(prefix + f"{k}: {formatted}")

        return "\n".join(lines)

    return str(value)


# Turn a document's fields into one plain-text block per field ("key: value", or "key:" followed
# by an indented sub-block for multi-line values), skipping skip_fields and any empty values.
def format_document(doc):

    output_lines = []

    for key, value in doc.items():

        if key in skip_fields:
            continue

        if value in ("", None, [], {}):
            continue

        formatted = format_value(value)

        if "\n" in formatted:
            output_lines.append(f"{key}:")
            output_lines.append(formatted)
        else:
            output_lines.append(f"{key}: {formatted}")

        output_lines.append("")

    return "\n".join(output_lines).strip()


In [5]:
# Convert every JSON file under sources/<source>/ into a cleaned .txt file under
# processed_sources/<source>/, mirroring the same folder layout. Non-JSON files (e.g. .DS_Store
# aside) are copied through as-is; empty or unparseable JSON files are skipped and counted.
# Already-converted files are skipped too, so if the kernel stops mid-run, re-running just
# resumes instead of redoing everything.
def convert_source(source):

    input_folder = f"sources/{source}"
    output_folder = f"processed_sources/{source}"

    converted = 0
    skipped = 0
    empty = 0
    invalid = 0
    i = 0

    for root, _, files in os.walk(input_folder):

        for file in files:

            i += 1

            input_path = os.path.join(root, file)
            relative = os.path.relpath(input_path, input_folder)

            # Skip macOS system files
            if file == ".DS_Store":
                continue

            # Copy non-JSON files through unchanged
            if not file.endswith(".json"):
                output_path = os.path.join(output_folder, relative)
                os.makedirs(os.path.dirname(output_path), exist_ok = True)
                shutil.copy2(input_path, output_path)
                continue

            output_path = os.path.join(output_folder, os.path.splitext(relative)[0] + ".txt")

            # Skip files already converted in a previous run
            if os.path.exists(output_path):
                skipped += 1
                continue

            # Skip empty files
            if os.path.getsize(input_path) == 0:
                print(f"Skipping empty file: {input_path}")
                empty += 1
                continue

            try:
                with open(input_path, "r", encoding = "utf-8") as f:
                    doc = json.load(f)

            except json.JSONDecodeError:
                print(f"Skipping invalid JSON: {input_path}")
                invalid += 1
                continue

            text = format_document(doc)

            os.makedirs(os.path.dirname(output_path), exist_ok = True)

            with open(output_path, "w", encoding = "utf-8") as f:
                f.write(text)

            converted += 1

            if i % 100 == 0:
                print(f"{source}: processed {i} files")

    print(f"Finished {source}! converted={converted} skipped={skipped} empty={empty} invalid={invalid}")


for source in sources:
    convert_source(source)


Finished confluence! converted=0 skipped=1000 empty=0 invalid=0
Finished fireflies! converted=0 skipped=1000 empty=0 invalid=0
Finished github! converted=0 skipped=1000 empty=0 invalid=0
Finished gmail! converted=0 skipped=1000 empty=0 invalid=0
Finished google_drive! converted=0 skipped=1000 empty=0 invalid=0
Finished hubspot! converted=0 skipped=1000 empty=0 invalid=0
Finished jira! converted=0 skipped=1000 empty=0 invalid=0
Finished linear! converted=0 skipped=1000 empty=0 invalid=0
Finished slack! converted=0 skipped=1000 empty=0 invalid=0
